- 2024/06/06
- Experiment to run multi-class classification using XGBoost.

In [1]:
#!/usr/bin/env python
#-*- coding:utf-8 -*-

import numpy as np
import os

# from xgboost import XGBClassifier, Booster
import xgboost as xgb
# from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score
# 250924: add the datetime to name the output folders with dates
from datetime import date

# 260209: add plt to visualize
import matplotlib.pyplot as plt

import sys
sys.path.append("..")
from data import Data
from options import Options
from xgbooster import XGBooster

/Users/huhao_lleida/Workplace/code_Postdoc_Lleida/ExplainingBoostedRules/Exp1_BoostRules_XGBoost/../xgbooster/xgbooster.py:498: SyntaxWarning: invalid escape sequence '\h'
  f.write("{} & {} & {} &{}  &{} & {} \\\\ \n \hline \n".format(


In [2]:
# 250924: prepare the name of the output folder with the date info
xgboost_output = 'temp_xgboost_' + date.today().isoformat()[2:].replace('-', '')

# 2025/6/5: trying XGBoost using xgbooster.py
# The default option split 20% as the testing set
bench_multi = '../bench_multi'
dataset_list = [os.path.join(bench_multi, n, n + '.csv') for n in sorted(os.listdir(bench_multi))]
xgb_nb_tree_candidate = list(range(10, 101, 10))
# 2025/9/24: add the candidated max_depth
xgb_max_depth = [4, 5, 6]

result_json = []

def train_specific_xgb(dataset, n_estimator, max_depth, output):
    # preparing the command for options
    command_line = 'xxx -t -n ' + str(n_estimator) + ' -d ' + str(max_depth) + ' -o ' + output + ' ' + dataset
    options = Options(command_line.split())

    if options.files:
        xgb = None
        if options.train:
            data = Data(filename=options.files[0], mapfile=options.mapfile,
                    separator=options.separator,
                    use_categorical=options.use_categorical)
            xgb = XGBooster(options, from_data=data)
            train_acc, test_acc, mod_fpath = xgb.train()
            return train_acc, test_acc, mod_fpath, xgb.nb_features, xgb.num_class, xgb.num_instance

In [3]:
## 2025/7/16 => function to extract the identical rules used in all boosted trees
import re
from collections import deque

def extract_info_tree_text_line(line, node_type):
    # node_type = 0 => internal node; node_type=1 => leaf node
    # DEBUG: considering the positive & negative for internal node
    assert node_type == 0 or node_type == 1
    line_v = line.split(':')
    assert len(line_v) == 2
    node_index = int(line_v[0])
    if node_type == 1:
        line_value = line_v[-1].split('=')[-1]
        line_dir_index = None
    if node_type == 0:
        line_value = line_v[-1].split(' ')[0]
        # only consider yes / no
        line_dirs = line_v[-1].split(' ')[-1].split(',')[:-1] 
        assert len(line_dirs) == 2
        line_dir_index = [-1, -1]
        for l_d in line_dirs:
            if l_d.split('=')[0] == 'yes':
                line_dir_index[0] = int(l_d.split('=')[-1])
            else:
                line_dir_index[1] = int(l_d.split('=')[-1])
    return node_index, line_value, line_dir_index
      
def dfs_tree_path(tree_text_lines, tree_index, exist_paths):
    '''
        dfs approach to iterate all paths in a given tree
    '''
    stack = deque()
    t_paths_name, t_paths = [], []
    for t_line in tree_text_lines:
        tab_level = len(t_line) - len(t_line.strip('\t'))
        while(len(stack) > tab_level):
            stack.pop()
        
        if 'leaf' in t_line:
            curr_path = []
            curr_path_name = 'path_t' + str(tree_index) + '['
            # need extract the current path, without push the leaf node
            last_dir = None
            for n in stack:
                n_index, l_value, l_dir = extract_info_tree_text_line(n, 0)
                curr_path_name += str(n_index)
                if (not last_dir is None) and (last_dir.index(n_index) == 1):
                    # judge the negation by the node_index
                    curr_path[-1] = '>='.join(curr_path[-1].split('<'))
                last_dir = l_dir
                curr_path.append(l_value)
        
            
            n_index, l_value, _ = extract_info_tree_text_line(t_line.strip('\t'), 1)
            curr_path_name += str(n_index) + ']:'
            if (not last_dir is None) and (last_dir.index(n_index) == 1):
                curr_path[-1] = '>='.join(curr_path[-1].split('<'))
            curr_path = ''.join(sorted(curr_path))
            # judge curr_path is identical or not
            if not (curr_path in exist_paths):
                # extract the leaf node value
                t_paths.append(''.join(curr_path))
                t_paths_name.append(curr_path_name)
            continue
        stack.append(t_line.strip('\t'))
    
    while(len(stack) > 0):
        stack.pop()
    assert len(stack) == 0
    return t_paths_name, t_paths

def extract_identical_paths_from_boosted_trees(booster_fpath, out_fpath):
    '''
        Input: the file path of boosted trees stored (in the default format of XGBoost package)
        Output: the txt containing all identical paths => same format as the boosted rules stored by BOOMER
    '''
    assert os.path.isfile(booster_fpath)
    identical_path_names, identical_paths = [], []

    with open(booster_fpath, 'r') as f:
        tree_index = -1
        tree_text = ''
        for line in f.readlines():
            if line.startswith('booster'):
                if len(tree_text) > 0:
                    t_text_lines = tree_text.split('\n')
                    t_paths_name, t_paths = dfs_tree_path(t_text_lines, tree_index, identical_paths)
                    identical_path_names += t_paths_name
                    identical_paths += t_paths
                    tree_text = ''
                # extract the index of tree 
                tree_index = int(re.search(r'\[(.*?)\]', line).group(1))
            else:
                tree_text += line
    
    # need to store the extracted paths into the out_fpath
    assert (len(identical_path_names) == len(identical_paths))
    with open(out_fpath, 'w') as f:
        for i in range(len(identical_paths)):
            f.write(identical_path_names[i] + '\n')
            f.write('{' + identical_paths[i] + '}\n')
    return len(identical_paths)


In [ ]:
for dataset in dataset_list:
    for n_estimator in xgb_nb_tree_candidate:
        for mdepth in xgb_max_depth:
            train_acc, test_acc, mod_fpath, n_f, n_c, n_i = train_specific_xgb(dataset, n_estimator, mdepth, xgboost_output)
    
            # 2025/7/17: add the extraction of paths operation of boosted trees
            mod_tpath_fpath = mod_fpath[:-3] + 'path'
            if os.path.exists(mod_tpath_fpath):
                os.remove(mod_tpath_fpath)
            num_identical_paths = extract_identical_paths_from_boosted_trees(mod_fpath, mod_tpath_fpath)
    
            dataset_n = dataset.split('/')[-1].split('.')[0]
            res_dict = {'dataset': dataset_n, 'n_estimator': n_estimator, 'max_depth': mdepth, 'train_acc': train_acc, \
                        'test_acc': test_acc, 'num_feat': n_f, 'num_class': n_c, 'num_instance': n_i, \
                        'num_identical_path': num_identical_paths}
            result_json.append(res_dict)


In [4]:
# 260209: just train XGBoost for Iris dataset, and visualize the tree structure
iris_csv = '../bench_multi/Iris.csv'
xgb_n, xgb_d = 2, 2

train_acc, test_acc, mod_fpath, _,_,_ = train_specific_xgb(iris_csv, xgb_n, xgb_d, 'iris_xgboost')
mod_tpath_fpath = mod_fpath[:-3] + 'path'
if os.path.exists(mod_tpath_fpath):
    os.remove(mod_tpath_fpath)
num_identical_paths = extract_identical_paths_from_boosted_trees(mod_fpath, mod_tpath_fpath)

In [ ]:
import json
json_file_name = 'result' + xgboost_output[4:] + '.json'
if os.path.isfile(json_file_name):
    os.remove(json_file_name)
with open(json_file_name, 'w') as f:
    json.dump(result_json, f, indent=2)